# 📊 Level 4: Data Cleaning, Scientific Analysis, and Visualization

## HydroSense-Kenya — From Raw Sensor Data to Scientific Insight

---

**Deliverables:**
1. Complete EDA with data quality assessment
2. Documented cleaning pipeline with audit trail
3. Outlier investigation and disposition
4. Statistical summaries per zone
5. **5+ publication-quality scientific visualisations** with interpretation
6. Cleaned dataset saved to `data/processed/`

In [ ]:
import sys, os
import warnings
warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

sys.path.insert(0, os.path.join('..', 'src'))
from data_cleaning import (
    clean_weather_data, clean_soil_data,
    detect_outliers_iqr, detect_outliers_zscore,
    compute_data_quality_score, validate_physical_ranges,
    flag_sensor_anomalies, PHYSICAL_BOUNDS,
)
from simulation import compute_et
from visualization import (
    setup_publication_style, COLORS, ZONE_COLORS,
    plot_weather_overview, plot_soil_moisture_zones,
)

setup_publication_style()

weather_raw = pd.read_csv('../data/raw/weather_daily.csv', na_values=['NA', ''])
soil_raw = pd.read_csv('../data/raw/soil_sensor_data.csv', na_values=['NA', ''])
params = pd.read_csv('../data/raw/crop_zone_parameters.csv')

print('Level 4: Data Analysis & Visualization — Setup complete ✓')
print(f'Weather: {weather_raw.shape}, Soil: {soil_raw.shape}, Params: {params.shape}')

---

## 1. Exploratory Data Analysis

In [ ]:
print('WEATHER DATA — Missing Values & Types')
print('=' * 55)
for col in weather_raw.columns:
    n_miss = weather_raw[col].isna().sum()
    pct = 100 * n_miss / len(weather_raw)
    dtype = weather_raw[col].dtype
    marker = ' ⚠️' if n_miss > 0 else ' ✓'
    print(f'  {col:<20} {dtype:<10} missing: {n_miss} ({pct:.1f}%){marker}')

print(f'\nSOIL SENSOR DATA — Missing Values')
print('=' * 55)
for col in soil_raw.columns:
    n_miss = soil_raw[col].isna().sum()
    pct = 100 * n_miss / len(soil_raw)
    marker = ' ⚠️' if n_miss > 0 else ' ✓'
    print(f'  {col:<25} missing: {n_miss} ({pct:.1f}%){marker}')

In [ ]:
print('WEATHER — Descriptive Statistics')
print('=' * 65)
print(weather_raw.describe().round(2).to_string())
print()
print('SOIL SENSORS — Descriptive Statistics by Zone')
print('=' * 65)
soil_raw['timestamp'] = pd.to_datetime(soil_raw['timestamp'])
for zone in ['Zone_A', 'Zone_B', 'Zone_C']:
    zd = soil_raw[soil_raw['zone_id'] == zone]
    crop = params[params['zone_id']==zone]['crop_type'].values[0]
    print(f'\n--- {zone} ({crop}) ---')
    print(zd[['soil_moisture_pct', 'tank_level_liters', 'pump_flow_lpm']].describe().round(2).to_string())

---

## 2. Outlier Investigation

In [ ]:
print('OUTLIER INVESTIGATION LOG')
print('=' * 75)
print()

# Temperature
print('1. Temperature 45.8°C on March 14')
print('   Kenya record ~41°C. Adjacent days: normal range.')
print('   Verdict: SENSOR FAULT — clip to 42°C')
print()

# Rainfall
print('2. Rainfall 85mm on March 26')
iqr_flags = detect_outliers_iqr(weather_raw['rainfall_mm'].dropna())
print('   IQR outlier detected. But 85mm is plausible for March heavy rains.')
print('   Verdict: RETAIN (flag only)')
print()

# Missing values
print('3. Missing rainfall on March 8 → Linear interpolation')
print('4. Missing humidity on March 21 → Linear interpolation')
print()

# Soil outliers
print('5. Tank level 9900 L (Zone C, March 14)')
print('   Tank capacity ~5000L. Physically impossible.')
print('   Verdict: SENSOR SPIKE — clip to 6000L')
print()
print('6. Pump flow 0.0 L/min with CHECK status (Zone B, March 21)')
print('   Verdict: PUMP FAULT — flag as anomaly')
print()
print('7. Soil moisture 8.5% (Zone B, March 25)')
print('   17.3 pct-point drop in one day is physically implausible.')
print('   Verdict: SENSOR FAULT — flag as anomaly')

---

## 3. Automated Cleaning Pipeline

In [ ]:
# Run cleaning pipeline
weather_clean, w_report = clean_weather_data(weather_raw)
soil_clean, s_report = clean_soil_data(soil_raw)

print('WEATHER CLEANING REPORT')
print(w_report.summary())
print()
print('SOIL SENSOR CLEANING REPORT')
print(s_report.summary())
print()

# Data quality scores
print('DATA QUALITY ASSESSMENT')
print('=' * 70)
w_quality = compute_data_quality_score(weather_raw, weather_clean)
print('\nWeather data quality:')
print(w_quality.to_string(index=False))

s_num = [c for c in soil_raw.select_dtypes(include=[np.number]).columns if c in soil_clean.columns]
s_quality = compute_data_quality_score(soil_raw, soil_clean, s_num)
print('\nSoil sensor data quality:')
print(s_quality.to_string(index=False))

---

## 4. Scientific Visualisations

### Visualisation 1: Weather Overview

In [ ]:
fig = plot_weather_overview(weather_clean)
plt.show()

print('📊 Interpretation:')
print('  • Rainfall is episodic with one extreme event (85mm on March 26)')
print('  • Temperature stable (22-27°C) with one clipped anomaly')
print('  • Humidity has a slight downward trend')

### Visualisation 2: Soil Moisture Dynamics by Zone

In [ ]:
fig = plot_soil_moisture_zones(soil_clean, params)
plt.show()

print('📊 Interpretation:')
print('  • Zone A (tomato): Starts highest, declining to stress zone')
print('  • Zone B (kale): Most variable, with anomalous dip')
print('  • Zone C (maize): Steadiest decline, below stress threshold by end')

### Visualisation 3: ET Component Decomposition

In [ ]:
T_c = weather_clean['temperature_c'].values
W_c = weather_clean['wind_speed_mps'].values
S_c = weather_clean['solar_index'].values
H_c = weather_clean['humidity_pct'].values
dates = pd.to_datetime(weather_clean['date'])

term_T = 0.12 * T_c
term_W = 0.35 * W_c
term_S = 2.4 * S_c
term_H = -0.025 * H_c
et_total = compute_et(T_c, W_c, S_c, H_c)

fig, ax = plt.subplots(figsize=(14, 6))
ax.fill_between(dates, 0, term_T, alpha=0.4, color=COLORS['red'],
                label=f'Temperature (mean={np.mean(term_T):.2f})')
ax.fill_between(dates, term_T, term_T + term_W, alpha=0.4, color=COLORS['green'],
                label=f'Wind (mean={np.mean(term_W):.2f})')
ax.fill_between(dates, term_T + term_W, term_T + term_W + term_S, alpha=0.4,
                color=COLORS['orange'], label=f'Solar (mean={np.mean(term_S):.2f})')
ax.plot(dates, et_total, color=COLORS['blue'], linewidth=2.5,
       label=f'Net ET (mean={np.mean(et_total):.2f})')
ax.axhline(0, color='black', linewidth=0.5)
ax.set_xlabel('Date')
ax.set_ylabel('ET contribution [mm/day]')
ax.set_title('Evapotranspiration Component Decomposition', fontweight='bold')
ax.legend(loc='upper right')
ax.xaxis.set_major_formatter(mdates.DateFormatter('%b %d'))
plt.tight_layout()
plt.show()

print(f'📊 Temperature contributes {np.mean(term_T):.2f} mm/day (dominant driver)')
print(f'   Net ET averages {np.mean(et_total):.2f} mm/day')

### Visualisation 4: Tank Level and Pump Flow

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 8), sharex=True)

for zone, color in ZONE_COLORS.items():
    zd = soil_clean[soil_clean['zone_id'] == zone]
    dates_s = pd.to_datetime(zd['timestamp'])
    axes[0].plot(dates_s, zd['tank_level_liters'], color=color, marker='o',
                 markersize=4, linewidth=1.5, label=zone)
    axes[1].plot(dates_s, zd['pump_flow_lpm'], color=color, marker='s',
                 markersize=4, linewidth=1.5, label=zone)

# Mark anomalies
if 'anomaly_flag' in soil_clean.columns:
    anomaly_rows = soil_clean[soil_clean['anomaly_flag'] == True]
    if len(anomaly_rows) > 0:
        anom_dates = pd.to_datetime(anomaly_rows['timestamp'])
        axes[0].scatter(anom_dates, anomaly_rows['tank_level_liters'],
                        color=COLORS['red'], s=100, zorder=5, marker='x',
                        linewidths=2, label='Anomaly')

axes[0].set_ylabel('Tank Level [litres]')
axes[0].set_title('Water Tank Level by Zone', fontweight='bold')
axes[0].legend()
axes[1].set_ylabel('Pump Flow [L/min]')
axes[1].set_xlabel('Date')
axes[1].set_title('Pump Flow Rate by Zone', fontweight='bold')
axes[1].legend()
axes[1].xaxis.set_major_formatter(mdates.DateFormatter('%b %d'))
fig.suptitle('Infrastructure Monitoring', fontsize=14, fontweight='bold', y=1.01)
fig.tight_layout()
plt.show()

### Visualisation 5: Correlation Heatmap

In [ ]:
soil_daily_avg = soil_clean.groupby(pd.to_datetime(soil_clean['timestamp']).dt.date).agg({
    'soil_moisture_pct': 'mean',
    'tank_level_liters': 'mean',
    'pump_flow_lpm': 'mean'
}).reset_index()
soil_daily_avg.columns = ['date', 'avg_moisture', 'avg_tank', 'avg_pump']
soil_daily_avg['date'] = pd.to_datetime(soil_daily_avg['date'])

weather_clean['date'] = pd.to_datetime(weather_clean['date'])
merged = pd.merge(weather_clean, soil_daily_avg, on='date', how='inner')

corr_cols = ['rainfall_mm', 'temperature_c', 'humidity_pct', 'wind_speed_mps',
             'solar_index', 'avg_moisture', 'avg_tank', 'avg_pump']
corr_matrix = merged[corr_cols].corr()

fig, ax = plt.subplots(figsize=(10, 8))
im = ax.imshow(corr_matrix, cmap='RdBu_r', vmin=-1, vmax=1, aspect='auto')
labels = ['Rainfall', 'Temperature', 'Humidity', 'Wind', 'Solar',
          'Avg Moisture', 'Avg Tank', 'Avg Pump']
ax.set_xticks(range(len(labels)))
ax.set_yticks(range(len(labels)))
ax.set_xticklabels(labels, rotation=45, ha='right')
ax.set_yticklabels(labels)
for i in range(len(labels)):
    for j in range(len(labels)):
        val = corr_matrix.iloc[i, j]
        color = 'white' if abs(val) > 0.5 else 'black'
        ax.text(j, i, f'{val:.2f}', ha='center', va='center', fontsize=9, color=color)
fig.colorbar(im, ax=ax, label='Pearson Correlation', shrink=0.8)
ax.set_title('Weather-Soil Correlation Matrix', fontweight='bold', fontsize=13)
plt.tight_layout()
plt.show()

---

## 5. Export Cleaned Dataset

In [ ]:
# Add ET column and save
et_daily = compute_et(
    weather_clean['temperature_c'].values,
    weather_clean['wind_speed_mps'].values,
    weather_clean['solar_index'].values,
    weather_clean['humidity_pct'].values,
)
weather_clean['et_mm'] = et_daily

weather_clean.to_csv('../data/processed/cleaned_irrigation_dataset.csv', index=False)
print('✓ Cleaned dataset saved to data/processed/cleaned_irrigation_dataset.csv')
print(f'  Shape: {weather_clean.shape}')
print(f'  Columns: {list(weather_clean.columns)}')
print(f'  Missing values remaining: {weather_clean.isna().sum().sum()}')

---

## 6. Summary

| Deliverable | Status |
||---|
| Full EDA with dtypes, nulls, distributions | ✅ |
| Cleaning pipeline with audit trail | ✅ |
| Outlier investigation (7 anomalies documented) | ✅ |
| Statistical summaries per zone | ✅ |
| Visualisation 1: Weather overview | ✅ |
| Visualisation 2: Soil moisture zones | ✅ |
| Visualisation 3: ET component decomposition | ✅ |
| Visualisation 4: Tank & pump diagnostics | ✅ |
| Visualisation 5: Correlation heatmap | ✅ |
| Cleaned dataset saved | ✅ |

**Next:** Level 5 builds the simulation and optimisation engine.